# Complex Particle Choi-Transfer Gap: `N20`, `Ny=30,40,50`

This A100 campaign uses the canonical GPU evolution entry point to propagate cumulative complex Choi blocks. At every completed cycle it extracts the particle-transfer Lyapunov gap from near-zero eigenstates of $A=\Sigma_{LL}$, equivalent to eigenstates of $\mathcal T_p\mathcal T_p^\dagger$ nearest eigenvalue one. Complete particle spectra and three near-gap eigenstates are saved only at $T=2N_y$.

In [9]:
from __future__ import annotations

from contextlib import contextmanager
import importlib.util
import inspect
import json
import sys
import threading
import time
from pathlib import Path
from typing import Any

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_regularized_choi_transfer_matrix'
# Set this to an absolute Drive path if the bundle is stored under a custom parent.
BUNDLE_ROOT_OVERRIDE = None  # e.g. '/content/drive/MyDrive/my_project/colab_regularized_choi_transfer_matrix'
def is_bundle_root(path: Path) -> bool:
    return ((path / 'src' / 'classA_U1FGTN_gpu.py').exists()
            and (path / 'src' / 'choi_transfer_observables_gpu.py').exists())

def find_bundle_root() -> Path:
    candidates = []
    if BUNDLE_ROOT_OVERRIDE is not None:
        candidates.append(Path(BUNDLE_ROOT_OVERRIDE).expanduser())
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base, base / BUNDLE_NAME])
    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        candidates.extend([drive_root / BUNDLE_NAME,
                           drive_root / 'class_A_fermionic_adaptive_circuit' / BUNDLE_NAME])
        candidates.extend(p for p in drive_root.rglob(BUNDLE_NAME) if p.is_dir())
    checked = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if is_bundle_root(candidate):
            return candidate
    raise FileNotFoundError(
        f'Could not locate {BUNDLE_NAME} with src/classA_U1FGTN_gpu.py and '
        'src/choi_transfer_observables_gpu.py. Upload the complete bundle to Drive '
        'or set BUNDLE_ROOT_OVERRIDE to its absolute path.'
    )

BUNDLE_ROOT = find_bundle_root()
SRC_ROOT = BUNDLE_ROOT / 'src'
ENGINE_PATH = SRC_ROOT / 'classA_U1FGTN_gpu.py'
OBSERVER_PATH = SRC_ROOT / 'choi_transfer_observables_gpu.py'

def load_bundle_module(module_name: str, module_path: Path):
    if not module_path.is_file():
        raise FileNotFoundError(f'Missing required bundle module: {module_path}')
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not create import specification for {module_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

HEARTBEAT_INTERVAL_S = 60.0


@contextmanager
def progress_heartbeat(progress_bar, config_label: str, *, interval_s: float = HEARTBEAT_INTERVAL_S):
    if interval_s <= 0:
        raise ValueError('heartbeat interval must be positive')
    stop_event = threading.Event()
    started = time.perf_counter()

    def emit() -> None:
        while not stop_event.wait(interval_s):
            elapsed_s = time.perf_counter() - started
            progress_bar.set_postfix_str(f'{config_label} elapsed={elapsed_s:.0f}s', refresh=True)
            progress_bar.write(f'[heartbeat] {config_label} still running; elapsed={elapsed_s:.0f}s')

    worker = threading.Thread(target=emit, name='colab-progress-heartbeat', daemon=True)
    worker.start()
    try:
        yield
    finally:
        stop_event.set()
        worker.join()
engine_module = load_bundle_module('classA_U1FGTN_gpu', ENGINE_PATH)
observer_module = load_bundle_module('choi_transfer_observables_gpu', OBSERVER_PATH)
classA_U1FGTN_gpu = engine_module.classA_U1FGTN_gpu
CANONICAL_ENTRY_POINT = observer_module.CANONICAL_ENTRY_POINT
CHOI_FORMULA_VERSION = observer_module.CHOI_FORMULA_VERSION
FINAL_SPECTRUM_EXTRACTION = observer_module.FINAL_SPECTRUM_EXTRACTION
GAP_DEFINITION = observer_module.GAP_DEFINITION
GAP_EXTRACTION = observer_module.GAP_EXTRACTION
HELPER_VERSION = observer_module.HELPER_VERSION
OBSERVABLE = observer_module.OBSERVABLE
PARTICLE_TRANSFER_DEFINITION = observer_module.PARTICLE_TRANSFER_DEFINITION
ParticleChoiGapObserver = observer_module.ParticleChoiGapObserver
exact_particle_spectrum_from_choi_blocks = observer_module.exact_particle_spectrum_from_choi_blocks
matrix_free_near_gap_single = observer_module.matrix_free_near_gap_single
save_npz_atomic = observer_module.save_npz_atomic
write_json_atomic = observer_module.write_json_atomic
if OBSERVABLE != 'complex_particle_choi_transfer_gap':
    raise RuntimeError(f'Unexpected observable helper: {OBSERVABLE}')
signature = inspect.signature(classA_U1FGTN_gpu.run_markov_circuit)
for name in ('track_choi', 'choi_observer', 'choi_observer_cycles', 'choi_singular_tol', 'choi_failure_mode', 'meas_slab_only'):
    if name not in signature.parameters:
        raise RuntimeError(f'Stale GPU engine missing {name}: {signature}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this campaign.')
torch.set_grad_enabled(False)
print(f'[setup] bundle={BUNDLE_ROOT}')
print(f'[setup] engine={ENGINE_PATH}')
print(f'[setup] observer={OBSERVER_PATH}')
print(f'[setup] helper={HELPER_VERSION}, gpu={torch.cuda.get_device_name(0)}')
if torch.cuda.get_device_properties(0).total_memory / 1024**3 < 35.0:
    print('[warn] Production parameters assume an A100 40 GB runtime.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[setup] bundle=/content/drive/MyDrive/colab_regularized_choi_transfer_matrix
[setup] engine=/content/drive/MyDrive/colab_regularized_choi_transfer_matrix/src/classA_U1FGTN_gpu.py
[setup] observer=/content/drive/MyDrive/colab_regularized_choi_transfer_matrix/src/choi_transfer_observables_gpu.py
[setup] helper=choi_transfer_observables_gpu_v5_particle_gap_censoring, gpu=NVIDIA A100-SXM4-40GB


In [10]:
NX = 20
NY_VALUES = [30, 40, 50]
SAMPLES = 25
NSHELL = 1
ALPHA_1 = 1.0
DTYPE, DEVICE, BACKEND = 'complex128', 'cuda:0', 'local'
SEQUENCE = 'raster_y'
PROTOCOL = 'perfect_correction'
BATCH_SIZE = 1
ENDPOINT_TOL = 1e-12
CHOI_SPECTRAL_TOL = 1e-10
CHOI_SINGULAR_TOL = 1e-10
CHOI_FAILURE_MODE = 'censor'
CHOI_ENTRY_ABS_TOL = 1e-10
CHOI_HERMITICITY_TOL = 1e-9
CHOI_INVOLUTION_TOL = 1e-8
N_EIGENSTATES = 3
NEAR_GAP_BLOCK_SIZE = 8
NEAR_GAP_MAX_SUBSPACE = 32
NEAR_GAP_MAX_ITER = 80
NEAR_GAP_RESIDUAL_TOL = 1e-10
OVERWRITE = True
SKIP_COMPLETED = True
DW_CONFIGS = [
    {'DW': True, 'dw_truncation': True, 'meas_slab_only': True, 'alpha_2': 30.0, 'label': 'DW1_dwtrunc1_slab_a2-30'},
    {'DW': False, 'dw_truncation': False, 'meas_slab_only': True, 'alpha_2': 1.0, 'label': 'DW0_full_a2-1'},
]
CONFIGS = [dict(Nx=NX, Ny=int(ny), cycles=2*int(ny), samples=SAMPLES, nshell=NSHELL,
                alpha_1=ALPHA_1, protocol=PROTOCOL, **dw)
           for ny in NY_VALUES for dw in DW_CONFIGS]
GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'complex_particle_choi_transfer'
CAMPAIGN_ID = 'N20_Ny30-50_nsh1_a1-1_S25_cycles2Ny'
CAMPAIGN_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

def config_id(cfg: dict[str, Any]) -> str:
    return f"N{cfg['Nx']}x{cfg['Ny']}_{cfg['label']}_nsh{cfg['nshell']}_{cfg['protocol']}"

def run_dir(cfg: dict[str, Any]) -> Path:
    return RUNS_ROOT / config_id(cfg)

def save_dataframe_atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

display(pd.DataFrame([{**cfg, 'config_id': config_id(cfg)} for cfg in CONFIGS]))


,Nx,Ny,cycles,samples,nshell,alpha_1,protocol,DW,dw_truncation,meas_slab_only,alpha_2,label,config_id
0,20,30,60,25,1,1.0,perfect_correction,True,True,True,30.0,DW1_dwtrunc1_slab_a2-30,N20x30_DW1_dwtrunc1_slab_a2-30_nsh1_perfect_co...
1,20,30,60,25,1,1.0,perfect_correction,False,False,True,1.0,DW0_full_a2-1,N20x30_DW0_full_a2-1_nsh1_perfect_correction
2,20,40,80,25,1,1.0,perfect_correction,True,True,True,30.0,DW1_dwtrunc1_slab_a2-30,N20x40_DW1_dwtrunc1_slab_a2-30_nsh1_perfect_co...
3,20,40,80,25,1,1.0,perfect_correction,False,False,True,1.0,DW0_full_a2-1,N20x40_DW0_full_a2-1_nsh1_perfect_correction
4,20,50,100,25,1,1.0,perfect_correction,True,True,True,30.0,DW1_dwtrunc1_slab_a2-30,N20x50_DW1_dwtrunc1_slab_a2-30_nsh1_perfect_co...
5,20,50,100,25,1,1.0,perfect_correction,False,False,True,1.0,DW0_full_a2-1,N20x50_DW0_full_a2-1_nsh1_perfect_correction


## Mandatory CUDA Smoke Test

The preflight checks exact-versus-matrix-free near-gap extraction, forced fallback, all adaptive protocol branches in slab-only mode, the homogeneous basis, Choi involution preservation, and denominator diagnostics.

In [11]:
def synthetic_solver_checks() -> None:
    torch.manual_seed(111)
    n = 18
    q, _ = torch.linalg.qr(torch.randn((n, n), dtype=torch.complex128, device=DEVICE))
    vals = torch.tensor([-1.0, -0.7, -0.03, -0.002, 0.001, 0.04, 0.3, 1.0] + [0.5] * 10,
                        dtype=torch.float64, device=DEVICE)
    a = q @ torch.diag(vals.to(torch.complex128)) @ q.mH
    exact = exact_particle_spectrum_from_choi_blocks(a.unsqueeze(0), cycle=7,
        endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3)
    estimate = matrix_free_near_gap_single(a, cycle=7, endpoint_tol=ENDPOINT_TOL,
        choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3, block_size=8,
        max_subspace=18, max_iter=80, residual_tol=1e-10, seed=5, exact_fallback=True)
    assert torch.allclose(estimate['near_gap_exponents'], exact['near_gap_exponents'][0], atol=1e-9, rtol=1e-8)
    forced = matrix_free_near_gap_single(a, cycle=7, n_eigenstates=3, block_size=8,
        max_subspace=18, max_iter=0, residual_tol=1e-14, seed=5, exact_fallback=True)
    assert bool(forced['used_exact_fallback'])
    assert torch.allclose(forced['gap'], exact['gap'][0])
    print('[smoke] synthetic matrix-free/exact and forced-fallback checks passed')

def run_protocol_smoke(protocol: str, *, dw: bool) -> None:
    samples = 1 if protocol == 'post_selection' else 2
    model = classA_U1FGTN_gpu(4, 8, DW=dw, nshell=1, alpha_1=1.0, alpha_2=30.0 if dw else 1.0,
                              dw_truncation=dw, device=DEVICE, dtype=DTYPE, backend=BACKEND)
    active = model.active_top_layer_indices(meas_slab_only=True)
    observer = ParticleChoiGapObserver(samples_expected=samples, cycles=range(1, 3), final_cycle=2,
        nlayer=int(active.numel()), n_eigenstates=3, near_gap_block_size=8, near_gap_max_subspace=24,
        near_gap_max_iter=20, near_gap_residual_tol=1e-9, endpoint_tol=ENDPOINT_TOL,
        choi_spectral_tol=CHOI_SPECTRAL_TOL, validate_choi=True, censor_unstable=True,
        choi_entry_abs_tol=CHOI_ENTRY_ABS_TOL, choi_hermiticity_tol=CHOI_HERMITICITY_TOL,
        choi_involution_tol=CHOI_INVOLUTION_TOL,
        active_top_layer_indices=active.detach().cpu().numpy(), full_nlayer=model.Nlayer)
    retained_sigma_ll = {}
    def smoke_observer(**payload):
        retained_sigma_ll[int(payload['cycle'])] = payload['sigma_ll'].detach().clone()
        return observer(**payload)
    kwargs = {'postselect': protocol == 'post_selection', 'perfect_correction': protocol == 'perfect_correction'}
    result = model.run_markov_circuit(G_history=False, progress=False, cycles=2, samples=samples,
        save=False, return_data=False, n_a=0.5, sequence=SEQUENCE, meas_slab_only=True, batch_size=samples,
        track_choi=True, choi_observer=smoke_observer, choi_observer_cycles=[1, 2],
        choi_singular_tol=CHOI_SINGULAR_TOL, choi_failure_mode=CHOI_FAILURE_MODE, **kwargs)
    observer.assert_complete()
    assert bool(result['meas_slab_only_effective']) == bool(dw)
    assert observer.gap.shape == (samples, 2)
    assert observer.stable_at_cycle.all()
    assert not observer.failure_details
    for cycle in (1, 2):
        exact = exact_particle_spectrum_from_choi_blocks(retained_sigma_ll[cycle], cycle=cycle,
            endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3)
        assert np.max(np.abs(observer.gap[:, cycle - 1] - exact['gap'].detach().cpu().numpy())) < 1e-8
    assert np.nanmax(observer.hermiticity_residual) < 1e-9
    assert np.nanmax(observer.involution_residual) < 1e-8
    assert np.max(observer.exact_vs_iterative_gap_error_final) < 1e-8
    assert min(row['min_abs_d'] for row in result['choi_diagnostics']) > CHOI_SINGULAR_TOL
    print(f'[smoke] DW={int(dw)} {protocol} gap observer passed')

synthetic_solver_checks()
for protocol in ('post_selection', 'perfect_correction', 'sampled_feedback'):
    run_protocol_smoke(protocol, dw=True)
run_protocol_smoke('perfect_correction', dw=False)
torch.cuda.empty_cache()


[smoke] synthetic matrix-free/exact and forced-fallback checks passed
[info] constructed Wannier projectors nshell=1 dw_truncation=True backend=local device=cuda:0 dtype=torch.complex128
[smoke] DW=1 post_selection gap observer passed
[info] constructed Wannier projectors nshell=1 dw_truncation=True backend=local device=cuda:0 dtype=torch.complex128
[smoke] DW=1 perfect_correction gap observer passed
[info] constructed Wannier projectors nshell=1 dw_truncation=True backend=local device=cuda:0 dtype=torch.complex128
[smoke] DW=1 sampled_feedback gap observer passed
[info] constructed Wannier projectors nshell=1 dw_truncation=False backend=local device=cuda:0 dtype=torch.complex128
[smoke] DW=0 perfect_correction gap observer passed


## Production Campaign

Each completed configuration is independently resumable through its `latest_run.json` marker. The engine evolves Choi blocks in memory and stores only cycle gaps, final exact spectra, final near-gap eigenstates, and diagnostics.

In [12]:
def expected_paths(out_dir: Path) -> dict[str, Path]:
    return {
        'gap': out_dir / 'particle_choi_transfer_gap_vs_cycle.npz',
        'spectrum': out_dir / 'particle_choi_transfer_final_spectrum.npz',
        'eigenstates': out_dir / 'particle_choi_transfer_final_eigenstates_near_gap.npz',
        'diagnostics': out_dir / 'particle_choi_transfer_diagnostics.npz',
        'summary': out_dir / 'run_summary.json',
        'latest': out_dir / 'latest_run.json',
    }

def completed_row(cfg: dict[str, Any], out_dir: Path) -> dict[str, Any] | None:
    latest = out_dir / 'latest_run.json'
    if not SKIP_COMPLETED or not latest.exists():
        return None
    payload = json.loads(latest.read_text())
    if payload.get('status') == 'completed' and payload.get('config_id') == config_id(cfg):
        summary_path = out_dir / 'run_summary.json'
        summary = json.loads(summary_path.read_text()) if summary_path.exists() else {}
        if summary.get('helper_version') == HELPER_VERSION and summary.get('choi_failure_mode') == CHOI_FAILURE_MODE:
            print(f'[skip] completed {config_id(cfg)}')
            return payload['run_index_row']
        print(f'[rerun] stale uncensored output found for {config_id(cfg)}')
    return None

def run_one_config(cfg: dict[str, Any]) -> dict[str, Any]:
    out_dir = run_dir(cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    old_row = completed_row(cfg, out_dir)
    if old_row is not None:
        return old_row
    cycles = int(cfg['cycles'])
    model = classA_U1FGTN_gpu(cfg['Nx'], cfg['Ny'], DW=cfg['DW'], nshell=cfg['nshell'],
        alpha_1=cfg['alpha_1'], alpha_2=cfg['alpha_2'], dw_truncation=cfg['dw_truncation'],
        device=DEVICE, dtype=DTYPE, backend=BACKEND)
    active = model.active_top_layer_indices(meas_slab_only=cfg['meas_slab_only'])
    n_active = int(active.numel())
    observer = ParticleChoiGapObserver(samples_expected=cfg['samples'], cycles=range(1, cycles + 1),
        final_cycle=cycles, nlayer=n_active, n_eigenstates=N_EIGENSTATES,
        near_gap_block_size=NEAR_GAP_BLOCK_SIZE, near_gap_max_subspace=NEAR_GAP_MAX_SUBSPACE,
        near_gap_max_iter=NEAR_GAP_MAX_ITER, near_gap_residual_tol=NEAR_GAP_RESIDUAL_TOL,
        endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, exact_fallback=True,
        censor_unstable=True, choi_entry_abs_tol=CHOI_ENTRY_ABS_TOL,
        active_top_layer_indices=active.detach().cpu().numpy(), full_nlayer=model.Nlayer)
    started = time.perf_counter()
    result = model.run_markov_circuit(G_history=False, progress=False, cycles=cycles, samples=cfg['samples'],
        postselect=False, perfect_correction=True, save=False, return_data=False, n_a=0.5,
        sequence=SEQUENCE, meas_slab_only=cfg['meas_slab_only'], batch_size=BATCH_SIZE, track_choi=True,
        choi_observer=observer, choi_observer_cycles=list(range(1, cycles + 1)),
        choi_singular_tol=CHOI_SINGULAR_TOL, choi_failure_mode=CHOI_FAILURE_MODE)
    elapsed_s = float(time.perf_counter() - started)
    observer.assert_complete()
    effective_slab = bool(cfg['DW'] and cfg['dw_truncation'] and cfg['meas_slab_only'])
    assert bool(result['meas_slab_only_effective']) == effective_slab
    assert observer.gap.shape == (cfg['samples'], cycles)
    assert observer.final_spectrum.shape == (cfg['samples'], n_active)
    metadata = {
        **cfg, 'config_id': config_id(cfg), 'actual_samples': cfg['samples'], 'Nlayer_active': n_active,
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT, 'observable': OBSERVABLE,
        'helper_version': HELPER_VERSION, 'choi_formula_version': CHOI_FORMULA_VERSION,
        'particle_transfer_definition': PARTICLE_TRANSFER_DEFINITION, 'transfer_object': 'T_p T_p^dagger',
        'final_eigenstate_index_convention': 'indices_into_sorted_final_spectrum',
        'gap_definition': GAP_DEFINITION, 'gap_extraction': GAP_EXTRACTION,
        'final_spectrum_extraction': FINAL_SPECTRUM_EXTRACTION, 'final_cycle_gap_override': 'mandatory_exact_eigh_Sigma_LL',
        'cycles_rule': 'cycles = 2 * Ny',
        'choi_basis': 'reduced_topological_slab' if effective_slab else 'full_top_layer',
        'meas_slab_only_requested': bool(cfg['meas_slab_only']), 'meas_slab_only_effective': effective_slab,
        'active_top_layer_indices': active.detach().cpu().numpy().tolist(), 'elapsed_s': elapsed_s,
        'solver': {'n_eigenstates': N_EIGENSTATES, 'block_size': NEAR_GAP_BLOCK_SIZE,
                   'max_subspace': NEAR_GAP_MAX_SUBSPACE, 'max_iter': NEAR_GAP_MAX_ITER,
                   'residual_tol': NEAR_GAP_RESIDUAL_TOL, 'exact_fallback_count': int(observer.used_exact_fallback.sum())},
        'endpoint_counts': 'evaluated only on exact fallback cycles and stable mandatory final exact cycle',
        'choi_failure_mode': CHOI_FAILURE_MODE, 'choi_singular_tol': CHOI_SINGULAR_TOL,
        'choi_entry_abs_tol': CHOI_ENTRY_ABS_TOL,
        'censoring_rule': 'exact Choi observable is NaN at and after first numerical-instability failure; physical G evolution continues',
        'stable_fraction_by_cycle': observer.stable_at_cycle.mean(axis=0).tolist(),
        'stable_samples_final': int(observer.stable_at_cycle[:, -1].sum()),
        'first_failure_cycle': observer.first_failure_cycle.tolist(),
        'first_failure_reason': observer.first_failure_reason.tolist(),
        'observer_failure_details': observer.failure_details,
        'choi_diagnostics': result['choi_diagnostics'], 'full_choi_or_transfer_matrices_saved': False,
    }
    paths = expected_paths(out_dir)
    if not OVERWRITE and any(path.exists() for path in paths.values()):
        raise FileExistsError(f'Output already exists for {config_id(cfg)}')
    metadata_json = np.asarray(json.dumps(metadata, sort_keys=True))
    save_npz_atomic(paths['gap'], **observer.gap_payload(), metadata_json=metadata_json)
    save_npz_atomic(paths['spectrum'], **observer.final_spectrum_payload(), metadata_json=metadata_json)
    save_npz_atomic(paths['eigenstates'], **observer.final_eigenstates_payload(), metadata_json=metadata_json)
    save_npz_atomic(paths['diagnostics'], **observer.diagnostics_payload(), metadata_json=metadata_json)
    write_json_atomic(paths['summary'], metadata)
    row = {'config_id': config_id(cfg), 'Nx': cfg['Nx'], 'Ny': cfg['Ny'], 'cycles': cycles,
           'DW': cfg['DW'], 'dw_truncation': cfg['dw_truncation'], 'samples': cfg['samples'],
           'Nlayer_active': n_active, 'stable_samples_final': int(observer.stable_at_cycle[:, -1].sum()),
           'elapsed_s': elapsed_s, 'output_dir': str(out_dir)}
    write_json_atomic(paths['latest'], {'status': 'completed', 'config_id': config_id(cfg), 'run_index_row': row})
    print(f"[production] {config_id(cfg)} completed in {elapsed_s:.1f}s; Nactive={n_active}; stable_final={int(observer.stable_at_cycle[:, -1].sum())}/{cfg['samples']}")
    del model, observer
    torch.cuda.empty_cache()
    return row


Markov GPU batches:   0%|          | 0/25 [19:57<?, ?sample/s]


In [ ]:
started = time.perf_counter()
rows = []
campaign_bar = tqdm(total=len(CONFIGS), desc='Choi-gap configurations', unit='config', leave=True, dynamic_ncols=True)
try:
    for cfg in CONFIGS:
        label = config_id(cfg)
        campaign_bar.set_postfix_str(f'{label} elapsed=0s', refresh=True)
        with progress_heartbeat(campaign_bar, label):
            rows.append(run_one_config(cfg))
        campaign_bar.update(1)
finally:
    campaign_bar.close()
run_index = pd.DataFrame(rows).sort_values(['Ny', 'DW']).reset_index(drop=True)
save_dataframe_atomic_csv(run_index, CAMPAIGN_ROOT / 'run_index.csv')
manifest = {
    'status': 'completed', 'campaign_name': CAMPAIGN_NAME, 'campaign_id': CAMPAIGN_ID,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT, 'observable': OBSERVABLE,
    'Nx': NX, 'Ny_values': NY_VALUES, 'cycles_rule': 'cycles = 2 * Ny', 'samples': SAMPLES,
    'configs': CONFIGS, 'results': rows, 'elapsed_s': float(time.perf_counter() - started),
    'campaign_root': str(CAMPAIGN_ROOT), 'run_index': str(CAMPAIGN_ROOT / 'run_index.csv'),
}
write_json_atomic(CAMPAIGN_ROOT / 'campaign_manifest.json', manifest)
write_json_atomic(CAMPAIGN_ROOT / 'latest_campaign.json', manifest)
display(run_index)
print(json.dumps(manifest, indent=2))


[info] constructed Wannier projectors nshell=1 dw_truncation=True backend=local device=cuda:0 dtype=torch.complex128
Markov GPU batches: 100%|██████████| 25/25 [1:19:53<00:00, 191.75s/sample]
[production] N20x30_DW1_dwtrunc1_slab_a2-30_nsh1_perfect_correction completed in 4793.8s; Nactive=660; stable_final=0/25
[info] constructed Wannier projectors nshell=1 dw_truncation=False backend=local device=cuda:0 dtype=torch.complex128
Markov GPU batches: 100%|██████████| 25/25 [2:25:15<00:00, 348.62s/sample]
[production] N20x30_DW0_full_a2-1_nsh1_perfect_correction completed in 8715.5s; Nactive=1200; stable_final=0/25
[info] constructed Wannier projectors nshell=1 dw_truncation=True backend=local device=cuda:0 dtype=torch.complex128
Markov GPU batches: 100%|██████████| 25/25 [2:37:25<00:00, 377.82s/sample]
[production] N20x40_DW1_dwtrunc1_slab_a2-30_nsh1_perfect_correction completed in 9445.4s; Nactive=880; stable_final=0/25
[info] constructed Wannier projectors nshell=1 dw_truncation=False ba

In [ ]:
from IPython.display import Javascript, display
try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
